## Example 1: Parse human-written durations

In [ ]:
import datetime
import importlib.metadata

import tempora

# Example 1: parse_timedelta reads "3 days, 4 hours", "1.5h", "2m 3s" and returns a timedelta
assert importlib.metadata.version("tempora") == "5.12.0"
assert tempora.parse_timedelta("3 days, 4 hours") == datetime.timedelta(days=3, hours=4)
assert tempora.parse_timedelta("1.5h") == datetime.timedelta(minutes=90)
assert tempora.parse_timedelta("2m 3s") == datetime.timedelta(minutes=2, seconds=3)
assert tempora.parse_timedelta("1 week") == datetime.timedelta(weeks=1)
print("Example 1 OK")

## Example 2: A portable strftime

In [ ]:
import datetime

import tempora

# Example 2: years before 1900, %s milliseconds and %µ microseconds work on every platform
assert tempora.strftime("%Y", datetime.datetime(900, 1, 1)) == "0900"
assert tempora.strftime("%Y-%m-%d", (1976, 5, 7)) == "1976-05-07"
assert tempora.strftime("%s", datetime.time(microsecond=20000)) == "020"
print("Example 2 OK")

## Example 3: Round and step through dates

In [ ]:
import datetime

import tempora

# Example 3: datetime_round to a period, and a date_range generator
t = datetime.datetime(2026, 11, 8, 2, 7, 45)
assert tempora.datetime_round(t, datetime.timedelta(minutes=15)) == datetime.datetime(2026, 11, 8, 2, 15)  # 7m45s rounds up
days = list(tempora.date_range(datetime.date(2026, 11, 1), datetime.date(2026, 11, 4)))
assert days == [datetime.date(2026, 11, d) for d in (1, 2, 3)], days
print("Example 3 OK")

## Example 4: Timers: Stopwatch and BackoffDelay

In [ ]:
import time

from tempora.timing import BackoffDelay, Stopwatch

# Example 4: a Stopwatch measures elapsed time; BackoffDelay grows a retry delay up to a limit
with Stopwatch() as sw:
    time.sleep(0.05)
assert 0.04 < sw.elapsed.total_seconds() < 5, sw.elapsed  # stopped on leaving the block
delay = BackoffDelay(delay=0.0, factor=2, limit=1, jitter=0)
delay.delay = 0.1
seen = [delay.delay]
for _ in range(4):
    next(delay)
    seen.append(delay.delay)
assert seen[-1] == 1 and seen[1] == 0.2, seen
print("Example 4 OK:", seen)

## Example 5: A scheduler for delayed and periodic commands

In [ ]:
import datetime

from tempora import schedule, utc

# Example 5: a command due now runs on run_pending(); a periodic one reschedules itself
ran = []
sched = schedule.CallbackScheduler(lambda target: ran.append(target()))
sched.add(schedule.DelayedCommand.after(0, lambda: "once"))
sched.add(schedule.PeriodicCommand.after(datetime.timedelta(hours=1), lambda: "hourly"))
sched.run_pending()
assert ran == ["once"], ran
assert utc.now().tzinfo is datetime.timezone.utc
print("Example 5 OK:", ran)